# 🛡️ UIDAI Risk Engine: Google Colab Training Pipeline

This notebook is designed to train the heavy Machine Learning and Deep Learning models (Isolation Forest, Dense Autoencoder, and LSTM) for the **UIDAI Anomaly & Risk Intelligence System** on high-performance Google Colab cloud infrastructure (CPU/GPU/TPU).

Once training is complete, the model checkpoints are saved directly to your Google Drive and can be downloaded and copied into the `models/` directory of your local project.

## Step 1: Install Dependencies
We install the required libraries for our training pipeline.

In [ ]:
!pip install pandas numpy scikit-learn tensorflow pyyaml joblib matplotlib seaborn

## Step 2: Mount Google Drive
Mount your Google Drive to load the raw datasets and save the trained model checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Define Project Paths
Adjust the paths below to match where you uploaded the `UIDAI-PROJECT-main` folder on your Google Drive.

In [ ]:
import os
import shutil

# Define paths on Drive
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/UIDAI-PROJECT-main"

if not os.path.exists(DRIVE_PROJECT_ROOT):
    print(f"⚠️ Drive project root {DRIVE_PROJECT_ROOT} not found. Please create it or copy files there.")
else:
    print(f"✓ Drive project root detected: {DRIVE_PROJECT_ROOT}")

## Step 4: Import Pipeline Modules
We add the project source folder to our python path and load the loader, features, and training modules.

In [ ]:
import sys
sys.path.append(DRIVE_PROJECT_ROOT)

from src.config import Config
from src.data.data_loader import DataLoader
from src.features.engineer import FeatureEngineer
from src.models.train import ModelTrainer

print("✓ Pipeline modules imported successfully.")

## Step 5: Execute Data Loading & Feature Engineering
Loads the full demographic and enrolment datasets, standardizes states, cleans duplicates, and extracts rolling window features.

In [ ]:
# Load configuration using the Drive path
config_path = os.path.join(DRIVE_PROJECT_ROOT, "config", "config.yaml")
config = Config(config_path=config_path)

# Override directories to point to Google Drive paths
loader = DataLoader(config)
loader.demo_dir = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.demographic_raw_dir"))
loader.enrol_dir = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.enrolment_raw_dir"))
loader.processed_dir = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.processed_dir"))

print("Loading datasets from Google Drive...")
merged_data = loader.run_pipeline()

print("Engineering features on the merged datasets...")
engineer = FeatureEngineer(config)
engineered_data = engineer.engineer_features(merged_data)
print(f"✓ Data prep complete. Ready to train. Dataset shape: {engineered_data.shape}")

## Step 6: Train ML and Deep Learning Models
This runs the Isolation Forest, Autoencoder (20 epochs), and LSTM network on the full dataset. All model weights will be saved directly back to Google Drive under the `models/` directory.

In [ ]:
trainer = ModelTrainer(config)
# Override training output paths to point to Drive
trainer.scaler_path = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.scaler_joblib"))
trainer.iforest_path = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.isolation_forest_joblib"))
trainer.autoencoder_path = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.autoencoder_keras"))
trainer.lstm_path = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.lstm_keras"))
trainer.models_dir = os.path.join(DRIVE_PROJECT_ROOT, config.get("paths.models_dir"))

print("Starting model training (Isolation Forest, Autoencoder, LSTM)... (This may take a few minutes on CPU/GPU)")
trainer.train_all(engineered_data)
print("✓ Training complete! All model checkpoints saved directly to Google Drive.")

## Step 7: Local Deployment Instructions
Once training is complete, follow these simple steps to deploy locally:
1. Open your Google Drive.
2. Navigate to `UIDAI-PROJECT-main/models/`.
3. Download the following files:
   - `scaler.joblib`
   - `isolation_forest.joblib`
   - `autoencoder.keras`
   - `lstm.keras`
4. Place these files in the local `models/` directory of your HP laptop.
5. Run the local Streamlit dashboard (`streamlit run app.py`) or FastAPI server. They will load the pre-trained checkpoints instantly without using CPU/RAM for training!